In [12]:
import pandas as pd



In [13]:
df = pd.read_csv('data/cropdata.csv')
df.head()

,crop ID,soil_type,Seedling Stage,MOI,temp,humidity,result
0,Wheat,Black Soil,Germination,1,25,80.0,1
1,Wheat,Black Soil,Germination,2,26,77.0,1
2,Wheat,Black Soil,Germination,3,27,74.0,1
3,Wheat,Black Soil,Germination,4,28,71.0,1
4,Wheat,Black Soil,Germination,5,29,68.0,1


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16411 entries, 0 to 16410
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   crop ID         16411 non-null  object 
 1   soil_type       16411 non-null  object 
 2   Seedling Stage  16411 non-null  object 
 3   MOI             16411 non-null  int64  
 4   temp            16411 non-null  int64  
 5   humidity        16411 non-null  float64
 6   result          16411 non-null  int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 897.6+ KB


In [15]:
# soil types
df['soil_type'].unique()

array(['Black Soil', 'Alluvial Soil', 'Sandy Soil', 'Red Soil',
       'Clay Soil', 'Loam Soil', 'Chalky Soil'], dtype=object)

In [16]:
# drop rows with 'Black Soil', 'Chalky Soil', 'Red Soil' and missing values
df_filtered = df[~df['soil_type'].isin(['Black Soil', 'Chalky Soil', 'Red Soil'])].drop(columns=['crop ID', 'Seedling Stage']).reset_index(drop=True).copy()
print(df_filtered['soil_type'].unique())
df_filtered.head()

['Alluvial Soil' 'Sandy Soil' 'Clay Soil' 'Loam Soil']


,soil_type,MOI,temp,humidity,result
0,Alluvial Soil,1,33,56.0,1
1,Alluvial Soil,2,34,53.0,1
2,Alluvial Soil,3,35,50.0,1
3,Alluvial Soil,4,36,47.0,1
4,Alluvial Soil,5,37,44.0,1


In [17]:
label_maping = {
    'Alluvial Soil': 0,
    'Sandy Soil': 1,
    'Clay Soil': 2,
    'Loam Soil': 3
}
df_filtered['soil_type'] = df_filtered['soil_type'].map(label_maping)
df_filtered.head()


,soil_type,MOI,temp,humidity,result
0,0,1,33,56.0,1
1,0,2,34,53.0,1
2,0,3,35,50.0,1
3,0,4,36,47.0,1
4,0,5,37,44.0,1


In [18]:
X = df_filtered.drop(columns=['result'])
y = df_filtered['result']

## Modelling

### SVM

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features (SVM is sensitive to feature scales)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train the SVM model
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_scaled, y_train)

# Make predictions using the test set
y_pred = svm_model.predict(X_test_scaled)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8530612244897959

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.95      0.89      1246
           1       0.87      0.85      0.86       825
           2       0.00      0.00      0.00       134

    accuracy                           0.85      2205
   macro avg       0.57      0.60      0.58      2205
weighted avg       0.80      0.85      0.83      2205



/Users/nickanthonymiras/miniconda3/envs/datascience/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/nickanthonymiras/miniconda3/envs/datascience/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/nickanthonymiras/miniconda3/envs/datascience/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this b

## Conversion

In [20]:
import pickle

In [21]:
with open("saved/scaler.pkl", 'wb') as f:
  pickle.dump(scaler, f)

In [22]:
with open("saved/svm_model.pkl", 'wb') as f:
  pickle.dump(svm_model, f)

# Conclusion